<a href="https://colab.research.google.com/github/ShabahangShayegan/TP1_compression_images_apprise_etudiant/blob/main/TP1_compression_images_apprise_etudiant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP1 — Compression d'images apprise : du compresseur jouet aux codecs de l'état de l'art

**Cours : Machine Learning appliqué à la compression vidéo — EPITA, majeure IA — Séance 1 (3h)**

| Partie | Durée | Contenu |
|---|---|---|
| **A** | ~30 min | Un compresseur appris **jouet**, écrit de A à Z en PyTorch sur des patchs d'images naturelles : relaxation de la quantification, perte $R + \lambda D$, **vrai codage rANS** |
| **B** | ~25 min | Deux codecs **pré-entraînés** de CompressAI : modèle factorisé (vrai flux binaire) et hyperprior + contexte (`cheng2020-anchor`), allocation des bits |
| **C** | ~40 min | Comparaison avec **JPEG, HEVC intra (x265)** et **VVC intra (VVenC)** : courbes R-D, **BD-rate**, complexité |
| **D** | ~10 min | Comparaison visuelle à bas débit |
| **E** | ~25 min | **Fine-tuning** d'un modèle pré-entraîné pour un nouveau compromis débit-distorsion |

Environ 2 h 10 de travail guidé : le reste de la séance sert à rédiger le rapport.

Les fonctions écrites ici (chargement, mesure du débit, PSNR/MS-SSIM, BD-rate, encodage x265/VVenC) sont réutilisables telles quelles sur vos propres images.

---
**Avant de commencer**
- `Fichier > Enregistrer une copie dans Drive`.
- `Exécution > Modifier le type d'exécution > GPU (T4)`. Tout fonctionne aussi sur CPU, plus lentement.
- Les cellules **✏️ Exercice** sont à compléter ; les **❓ Questions** vont dans le rapport de votre équipe.
- Les résultats longs à calculer sont **sauvegardés sur votre Drive** : après une coupure de session, ré-exécutez simplement les cellules.

## 0. Configuration et installation

CompressAI fixe `numpy<2` dans ses dépendances, ce qui forcerait Colab à rétrograder numpy et à redémarrer. La bibliothèque fonctionne très bien avec numpy 2 : on l'installe **sans ses dépendances** et on ajoute à la main les paquets manquants.

In [ ]:
# ===================== Paramètres du TP =====================
ASSETS_REPO = "ShabahangShayegan/epita-ml-video-assets"   # dépôt Hugging Face (type dataset) de l'enseignant
USE_DRIVE   = True     # sauvegarde des résultats sur Google Drive (reprise après coupure)
USE_PRECOMPUTED_IF_MISSING = True   # résultats de l'enseignant si un codec ne peut pas tourner
RUN_VVENC   = True     # VVC intra (VVenC) comme ancre supplémentaire
# ============================================================

In [ ]:
!pip install -q --no-deps compressai==1.2.8 torch-geometric pytorch-msssim einops
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -E "libx265|libwebp" || echo "ATTENTION : encodeur manquant"

In [ ]:
import io, os, sys, re, glob, json, time, shutil, subprocess, tempfile, warnings, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pytorch_msssim import ms_ssim
from scipy.interpolate import PchipInterpolator

from compressai.zoo import bmshj2018_factorized, cheng2020_anchor
from compressai.ans import RansEncoder, RansDecoder
from compressai._CXX import pmf_to_quantized_cdf

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "pas de GPU (ça marche, mais plus lentement)")
torch.manual_seed(0); np.random.seed(0)
PRETRAINED = True    # poids pré-entraînés téléchargés depuis le dépôt CompressAI
plt.rcParams["figure.dpi"] = 110

IN_COLAB = "google.colab" in sys.modules
WORK = "tp1_work"
if IN_COLAB:
    WORK = "/content/tp1_work"
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            WORK = "/content/drive/MyDrive/EPITA_TP1_images"
        except Exception as e:
            print("Drive non monté, résultats en local seulement :", e)
LOCAL = "/content/tp1_local" if IN_COLAB else os.path.join(WORK, "local")
for d in [os.path.join(WORK, s) for s in ["data", "results", "bin", "assets"]] + [LOCAL]:
    os.makedirs(d, exist_ok=True)
print("Dossier de travail :", WORK)

def sh(cmd, check=True, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if check and r.returncode != 0:
        raise RuntimeError(f"Échec ({r.returncode}) : {cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout + r.stderr

def hf_get(path_in_repo):
    # Télécharge un fichier du dépôt de ressources de l'enseignant (None si indisponible)
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(ASSETS_REPO, path_in_repo, repo_type="dataset", local_dir=os.path.join(WORK, "assets"))
    except Exception as e:
        print(f"  [ressource indisponible] {path_in_repo} : {type(e).__name__}")
        return None

## 1. Données : le jeu Kodak

Le jeu **Kodak** (24 images naturelles 768×512) est l'ensemble de test historique de la compression d'images : presque tous les articles du domaine y rapportent leurs résultats.

Les images 1 à 16 servent à l'**entraînement** (compresseur jouet, partie A, et fine-tuning, partie E) ; **les images 17 à 24 forment l'ensemble de test** de toutes les mesures.

In [ ]:
KODAK_URL = "http://r0k.us/graphics/kodak/kodak/kodim{:02d}.png"

def load_kodak():
    imgs, use_hf = [], True
    try:
        for i in range(1, 25):
            path = os.path.join(WORK, "data", f"kodim{i:02d}.png")
            if not os.path.exists(path):
                src = hf_get(f"kodak/kodim{i:02d}.png") if use_hf else None
                if src:
                    shutil.copy(src, path)
                else:          # dépôt de l'enseignant indisponible : site d'origine
                    use_hf = False
                    urllib.request.urlretrieve(KODAK_URL.format(i), path)
            imgs.append(np.array(Image.open(path).convert("RGB")))
        print("Kodak chargé : 24 images")
    except Exception as e:
        print("Kodak indisponible (", e, ") -> images de repli scikit-image : LES CONCLUSIONS NE VALENT PAS POUR KODAK")
        from skimage import data
        base = [data.astronaut(), data.coffee(), data.chelsea(), data.rocket()]
        imgs = [np.array(Image.fromarray(b).resize((768, 512))) for b in base for _ in range(6)]
    return imgs

kodak = load_kodak()
train_imgs, test_imgs = kodak[:16], kodak[16:]

fig, axes = plt.subplots(2, 4, figsize=(12, 4.2))
for ax, im, k in zip(axes.ravel(), test_imgs, range(17, 25)):
    ax.imshow(im); ax.set_title(f"kodim{k}", fontsize=8); ax.axis("off")
plt.suptitle("Ensemble de test"); plt.tight_layout(); plt.show()

### Utilitaires : conversion, padding, métriques

In [ ]:
def to_tensor(img_uint8):
    # HxWx3 uint8 -> 1x3xHxW float dans [0,1]
    return torch.from_numpy(img_uint8).permute(2, 0, 1).float().div(255.).unsqueeze(0)

def to_uint8(x):
    return (x.clamp(0, 1).squeeze(0).permute(1, 2, 0).mul(255.).round().byte().cpu().numpy())

def pad(x, p=64):
    # complète l'image (réplication des bords) jusqu'à un multiple de p ; renvoie aussi la taille d'origine
    h, w = x.shape[-2:]
    return F.pad(x, (0, (-w) % p, 0, (-h) % p), mode="replicate"), (h, w)

def crop(x, size):
    return x[..., :size[0], :size[1]]

def psnr(a_uint8, b_uint8):
    mse = np.mean((a_uint8.astype(np.float64) - b_uint8.astype(np.float64)) ** 2)
    return 10 * np.log10(255.0 ** 2 / mse)

def msssim(a_uint8, b_uint8):
    return ms_ssim(to_tensor(a_uint8), to_tensor(b_uint8), data_range=1.0).item()

def msssim_db(v):
    # MS-SSIM en dB, plus lisible sur les courbes : -10 log10(1 - MS-SSIM)
    return -10 * np.log10(1 - v)

def n_pixels(img):
    return img.shape[0] * img.shape[1]

def codec_pil(img, fmt, quality):
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format=fmt, quality=quality)
    n = buf.tell(); buf.seek(0)
    return n, np.array(Image.open(buf).convert("RGB"))

---
# Partie A — Un compresseur appris jouet, de bout en bout (~30 min)

Avant d'utiliser des modèles de l'état de l'art, construisons le plus petit compresseur appris complet possible, **sur des patchs 32×32 d'images naturelles** :

$$x \xrightarrow{\;g_a\;} y \xrightarrow{\;Q\;} \hat y \xrightarrow{\;\text{rANS}\;} \texttt{0110…} \xrightarrow{\;\text{rANS}^{-1}\;} \hat y \xrightarrow{\;g_s\;} \hat x$$

- $g_a$ / $g_s$ : transformées d'analyse et de synthèse (trois convolutions de pas 2) ;
- **modèle de probabilité** : une gaussienne centrée $\mathcal{N}(0, \sigma_c^2)$ par canal $c$ du latent, avec $\sigma_c$ **appris** ;
- **débit** d'un élément quantifié : $-\log_2 P(\hat y)$ avec $P(\hat y) = \Phi\!\left(\frac{\hat y + 1/2}{\sigma_c}\right) - \Phi\!\left(\frac{\hat y - 1/2}{\sigma_c}\right)$ ;
- **apprentissage** : l'arrondi a un gradient nul presque partout. On le remplace pendant l'entraînement par l'**ajout d'un bruit uniforme** $\tilde y = y + u$, $u \sim \mathcal{U}(-\frac12, \frac12)$.

### ✏️ Exercice 1 — Débit estimé sous le modèle gaussien

Complétez `rate_bits(y, sigma)` : nombre total de bits $\sum_i -\log_2 P(y_i)$ selon la formule ci-dessus (`sigma` est diffusable sur `y`). Bornez la probabilité inférieurement par $10^{-9}$ pour éviter $\log 0$.

In [ ]:
STD_NORMAL = torch.distributions.Normal(0.0, 1.0)

def rate_bits(y, sigma):
    # TODO : probabilité de l'intervalle [y-1/2, y+1/2] sous N(0, sigma²) (utilisez STD_NORMAL.cdf),
    #        bornée par 1e-9, puis somme des -log2
    raise NotImplementedError("Exercice 1")

In [ ]:
class ToyCodec(nn.Module):
    def __init__(self, C=16, N=64):
        super().__init__()
        self.g_a = nn.Sequential(
            nn.Conv2d(3, N, 5, 2, 2), nn.LeakyReLU(),
            nn.Conv2d(N, N, 5, 2, 2), nn.LeakyReLU(),
            nn.Conv2d(N, C, 5, 2, 2))
        self.g_s = nn.Sequential(
            nn.ConvTranspose2d(C, N, 5, 2, 2, output_padding=1), nn.LeakyReLU(),
            nn.ConvTranspose2d(N, N, 5, 2, 2, output_padding=1), nn.LeakyReLU(),
            nn.ConvTranspose2d(N, 3, 5, 2, 2, output_padding=1))
        self.log_sigma = nn.Parameter(torch.zeros(C))   # sigma_c = exp(log_sigma_c)

    def sigma(self):
        return self.log_sigma.exp().view(1, -1, 1, 1)

    def forward(self, x, noise=True):
        y = self.g_a(x)
        y_tilde = y + torch.empty_like(y).uniform_(-0.5, 0.5) if noise else torch.round(y)
        return {"x_hat": self.g_s(y_tilde), "y": y, "y_tilde": y_tilde, "bits": rate_bits(y_tilde, self.sigma())}

toy = ToyCodec()
x = to_tensor(train_imgs[0][:32, :32])
out = toy(x)
print("x :", tuple(x.shape), "-> y :", tuple(out["y"].shape), f"| {sum(p.numel() for p in toy.parameters())/1e3:.0f} k paramètres")

**❓ Question**
1. Pourquoi ajoute-t-on un bruit uniforme pendant l'entraînement au lieu d'arrondir ? Pourquoi un bruit de largeur exactement 1 ?

### Entraînement : minimiser $R + \lambda \cdot 255^2 \cdot \mathrm{MSE}$

On entraîne trois modèles avec trois valeurs de $\lambda$, sur des patchs 32×32 tirés au hasard dans les images 1 à 16. Le débit est exprimé en **bits par pixel (bpp)**.

⏱️ Environ 1 minute par modèle sur GPU T4 (3 à 4 minutes sur CPU).

In [ ]:
TOY_LAMBDAS = [0.002, 0.01, 0.05]
TOY_ITERS = 3000 if device == "cuda" else 1500
PATCH, TOY_BATCH = 32, 64 if device == "cuda" else 32

train_t = [to_tensor(im)[0] for im in train_imgs]
def random_patches(n, p, src=train_t):
    out = []
    for _ in range(n):
        t = src[np.random.randint(len(src))]
        i = np.random.randint(t.shape[1] - p + 1); j = np.random.randint(t.shape[2] - p + 1)
        out.append(t[:, i:i + p, j:j + p])
    return torch.stack(out).to(device)

rng_state = np.random.get_state(); np.random.seed(123)
test_patches = random_patches(512, PATCH, [to_tensor(im)[0] for im in test_imgs])
np.random.set_state(rng_state)

def train_toy(lmbda):
    torch.manual_seed(0)
    m = ToyCodec().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, TOY_ITERS)
    for it in range(TOY_ITERS):
        x = random_patches(TOY_BATCH, PATCH)
        o = m(x)
        bpp = o["bits"] / (x.shape[0] * PATCH * PATCH)
        loss = bpp + lmbda * 255 ** 2 * F.mse_loss(o["x_hat"], x)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return m.eval()

def eval_toy_patches(m):
    with torch.no_grad():
        o = m(test_patches, noise=False)
    bpp = o["bits"].item() / (test_patches.shape[0] * PATCH * PATCH)
    return bpp, (10 * torch.log10(1 / F.mse_loss(o["x_hat"].clamp(0, 1), test_patches))).item()

toys = {}
for lm in TOY_LAMBDAS:
    t0 = time.time(); toys[lm] = train_toy(lm)
    bpp, p = eval_toy_patches(toys[lm])
    print(f"λ={lm:<6} : {bpp:.3f} bpp, PSNR {p:.2f} dB  (σ appris : {toys[lm].log_sigma.exp().detach().cpu().numpy().round(2)})  [{time.time()-t0:.0f} s]")

### ✏️ Exercice 2 — Tables de fréquences pour le codeur entropique

Le codeur **rANS** ne manipule pas de flottants : il utilise des **CDF entières** (précision 16 bits), identiques au bit près à l'encodeur et au décodeur. Pour chaque canal $c$ :
1. support $\{-L_c, \dots, L_c\}$ avec $L_c = \lceil 6\sigma_c \rceil + 1$ ;
2. probabilités $P(k)$ de chaque entier $k$ du support (même formule qu'à l'exercice 1), plus la **masse de queue** $1 - \sum_k P(k)$ ajoutée en dernier symbole (« échappement ») ;
3. conversion en CDF entière avec `pmf_to_quantized_cdf(liste_de_probas, 16)`.

Complétez `build_cdf_tables(sigmas)`, qui renvoie `cdfs` (liste de listes d'entiers), `cdf_lengths` (= taille du support + 2) et `offsets` (= $-L_c$).

In [ ]:
def build_cdf_tables(sigmas):
    cdfs, lengths, offsets = [], [], []
    for s in sigmas:
        # TODO : support, pmf, masse de queue, CDF entière ; remplir les trois listes
        raise NotImplementedError("Exercice 2")
    return cdfs, lengths, offsets

In [ ]:
class ToyCompressor:
    # Encodeur et décodeur complets : image -> octets -> image
    def __init__(self, model):
        self.m = model
        self.tables = build_cdf_tables(model.log_sigma.exp().tolist())

    def compress(self, img):
        x, size = pad(to_tensor(img).to(device), 8)
        with torch.no_grad():
            y_hat = torch.round(self.m.g_a(x))
        C, h, w = y_hat.shape[1:]
        idx = torch.arange(C).view(-1, 1, 1).expand(C, h, w).flatten().tolist()
        data = RansEncoder().encode_with_indexes(y_hat.int().flatten().tolist(), idx, *self.tables)
        return data, (C, h, w, size), y_hat

    def decompress(self, data, meta):
        C, h, w, size = meta
        idx = torch.arange(C).view(-1, 1, 1).expand(C, h, w).flatten().tolist()
        vals = RansDecoder().decode_with_indexes(data, idx, *self.tables)
        y_hat = torch.tensor(vals, dtype=torch.float32, device=device).view(1, C, h, w)
        with torch.no_grad():
            return to_uint8(crop(self.m.g_s(y_hat), size))

img = test_imgs[6]
tc = ToyCompressor(toys[TOY_LAMBDAS[1]])
data, meta, y_hat = tc.compress(img)
rec = tc.decompress(data, meta)
est = rate_bits(y_hat, tc.m.sigma()).item()
L_sup = torch.tensor([-o for o in tc.tables[2]], device=device).view(1, -1, 1, 1)
print(f"Latent : {meta[:3]} | flux réel : {len(data)} octets = {8*len(data)/n_pixels(img):.3f} bpp | estimé : {est/8:.0f} octets")
print(f"Valeurs hors support (codées par échappement) : {100*(y_hat.abs() > L_sup).float().mean().item():.2f} %")
with torch.no_grad():
    rec_enc = to_uint8(crop(tc.m.g_s(y_hat), meta[3]))
print("Décodage identique à la reconstruction de l'encodeur :", np.array_equal(rec, rec_enc))

# JPEG au débit le plus proche
q_best = min(range(2, 96), key=lambda q: abs(codec_pil(img, "JPEG", q)[0] - len(data)))
nj, rj = codec_pil(img, "JPEG", q_best)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].imshow(img); ax[0].set_title("original")
ax[1].imshow(rec); ax[1].set_title(f"compresseur jouet — {8*len(data)/n_pixels(img):.3f} bpp, {psnr(img, rec):.2f} dB")
ax[2].imshow(rj); ax[2].set_title(f"JPEG q={q_best} — {8*nj/n_pixels(img):.3f} bpp, {psnr(img, rj):.2f} dB")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Mini courbe R-D du compresseur jouet sur les 8 images de test, comparée à JPEG
rows = []
for lm, m in toys.items():
    t = ToyCompressor(m)
    for im in test_imgs:
        d, meta, _ = t.compress(im); r = t.decompress(d, meta)
        rows.append(dict(codec="jouet", param=lm, bpp=8 * len(d) / n_pixels(im), psnr=psnr(im, r)))
for q in [5, 10, 20, 35, 50, 75, 90]:
    for im in test_imgs:
        n, r = codec_pil(im, "JPEG", q); rows.append(dict(codec="JPEG", param=q, bpp=8 * n / n_pixels(im), psnr=psnr(im, r)))
toy_rd = pd.DataFrame(rows).groupby(["codec", "param"]).mean().reset_index()
plt.figure(figsize=(6.5, 4.2))
for c, g in toy_rd.groupby("codec"):
    g = g.sort_values("bpp"); plt.plot(g.bpp, g.psnr, "o-" if c == "jouet" else "s--", label=c)
plt.xlabel("bpp"); plt.ylabel("PSNR RGB (dB)"); plt.grid(alpha=.3); plt.legend(); plt.title("Compresseur jouet vs JPEG (Kodak 17-24)")
plt.show()

**❓ Questions**
2. Comparez débit réel (rANS) et débit estimé. D'où vient l'écart ? Pourquoi une valeur hors support, codée par **échappement**, peut-elle coûter moins que $-\log_2 P(\hat y)$ ?
3. Que se passerait-il si l'encodeur et le décodeur calculaient leurs tables de fréquences **en flottants**, sur deux machines différentes ?
4. Le compresseur jouet bat-il JPEG ? Citez trois limites de ce modèle qu'un codec moderne corrige (GDN, hyperprior, modèle de contexte, profondeur…).

---
# Partie B — Les codecs pré-entraînés de CompressAI (~25 min)

[CompressAI](https://github.com/InterDigitalInc/CompressAI) (InterDigital) fournit les modèles de référence du domaine, pré-entraînés à 6 à 8 niveaux de qualité, chacun avec un λ différent dans $L = R + \lambda \cdot 255^2 \cdot \mathrm{MSE}$. Nous en utilisons deux, aux deux extrémités de l'évolution vue en cours :

| Modèle | Modèle d'entropie | Évaluation |
|---|---|---|
| **`bmshj2018-factorized`** (Ballé et al., 2018) | une densité apprise par canal, comme le compresseur jouet en plus riche | **vrai flux binaire** |
| **`cheng2020-anchor`** (Cheng et al., 2020) | hyperprior + modèle de contexte autorégressif | **débit estimé** (décodage séquentiel trop lent) |

Deux modes d'utilisation :
- `net(x)` (forward) : celui de l'**entraînement**, renvoie la reconstruction et les **vraisemblances** des latents (débit *estimé*) ;
- `net.compress(x)` / `net.decompress(...)` : quantification + **codage entropique réel** (rANS), renvoie des **octets**.

In [ ]:
# Codage entropique sur CPU : encodeur et décodeur doivent calculer exactement les mêmes tables (voir l'encadré plus bas).
ENTROPY_DEVICE = "cpu"

def roundtrip(model, x):
    # compress + decompress sur ENTROPY_DEVICE. Renvoie (flux, x_hat, t_enc, t_dec, écart encodeur/décodeur).
    dev = next(model.parameters()).device
    m, xx = model.to(ENTROPY_DEVICE), x.to(ENTROPY_DEVICE)
    with torch.no_grad():
        t0 = time.time(); e = m.compress(xx); t1 = time.time()
        d = m.decompress(e["strings"], e["shape"]); t2 = time.time()
        ref = m(xx)["x_hat"]          # reconstruction vue par l'encodeur (mode eval : arrondi, pas de bruit)
    gap = (d["x_hat"] - ref).abs().max().item()
    model.to(dev)
    return e, d["x_hat"].to(dev), t1 - t0, t2 - t1, gap

net = bmshj2018_factorized(quality=3, pretrained=PRETRAINED).eval().to(device)
net.update(force=True)   # construit les tables CDF entières (comme l'exercice 2, mais pour des densités apprises)

img = test_imgs[6]       # kodim23 (les perroquets)
x_pad, size = pad(to_tensor(img).to(device))

enc, x_hat, _, _, gap = roundtrip(net, x_pad)
rec = to_uint8(crop(x_hat, size))
n_bytes = sum(len(s[0]) for s in enc["strings"])
bpp_real = 8 * n_bytes / n_pixels(img)
print(f"Écart max. encodeur/décodeur : {gap:.1e} (doit être ~0)")
print(f"Flux : {n_bytes} octets  ->  {bpp_real:.3f} bpp   (image brute : 24 bpp, soit un facteur {24/bpp_real:.0f})")
print(f"PSNR : {psnr(img, rec):.2f} dB | MS-SSIM : {msssim(img, rec):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(img); ax[0].set_title("Original"); ax[1].imshow(rec); ax[1].set_title(f"bmshj2018-factorized q=3 — {bpp_real:.3f} bpp")
for a in ax: a.axis("off")
plt.show()

> **⚠️ Déterminisme.** Le décodeur doit retrouver **exactement** les tables de l'encodeur. Avec un hyperprior, il recalcule les $\sigma$ à partir de $\hat z$ ; sur GPU, un écart au dernier bit suffit à corrompre l'image décodée (le débit, lui, reste normal). `roundtrip` code donc sur CPU et vérifie l'écart encodeur/décodeur.

### ✏️ Exercice 3 — Débit estimé à partir des vraisemblances

Généralisez l'exercice 1 : `out["likelihoods"]` est un dictionnaire de tenseurs de probabilités (`"y"` seul pour un modèle factorisé ; `"y"` et `"z"` pour un modèle avec hyperprior). Complétez `estimated_bpp(out, num_pixels)` ; le résultat doit être un **tenseur scalaire différentiable** (il resservira pour le fine-tuning).

In [ ]:
def estimated_bpp(out, num_pixels):
    # TODO : somme des -log2(vraisemblance) sur tous les tenseurs, divisée par le nombre de pixels
    raise NotImplementedError("Exercice 3")

In [ ]:
with torch.no_grad():
    out = net(x_pad)
bpp_est = estimated_bpp(out, n_pixels(img)).item()
print(f"Débit estimé (vraisemblances) : {bpp_est:.4f} bpp")
print(f"Débit réel (flux rANS)        : {bpp_real:.4f} bpp")
print(f"Écart relatif                 : {100*(bpp_real-bpp_est)/bpp_est:+.2f} %")

### ✏️ Exercice 4 — Carte d'allocation des bits

Calculez une carte 2D du **nombre de bits dépensés à chaque position** du latent $y$ (somme de $-\log_2 p$ sur les canaux). L'affichage est fourni.

In [ ]:
def bit_allocation_map(likelihoods_y):
    # likelihoods_y : 1 x M x h x w  ->  carte h x w (numpy) en bits
    # TODO
    raise NotImplementedError("Exercice 4")

### Hyperprior et contexte : `cheng2020-anchor`

| | rôle |
|---|---|
| $h_a$ (hyper-analyse) | $y$ → hyper-latent $z$ (4 fois plus petit encore), codé avec un modèle factorisé (flux `z`) |
| $h_s$ (hyper-synthèse) | $\hat z$ → paramètres de la loi de **chaque élément** de $y$ |
| contexte | convolution masquée sur les éléments de $\hat y$ **déjà décodés** |
| paramètres | combinaison des deux → moyenne $\mu$ et écart-type $\sigma$ de chaque élément |

Le $\sigma$ n'est plus fixe par canal comme dans le modèle factorisé : il est **prédit pour chaque position de chaque image**. On compare les deux modèles sur la même image, au même niveau de qualité, en débit **estimé** pour les deux (pour comparer à méthode égale).

In [ ]:
n_params = lambda m: sum(p.numel() for p in m.parameters())
cheng = cheng2020_anchor(quality=3, pretrained=PRETRAINED).eval().to(device)
with torch.no_grad():
    oc = cheng(x_pad)
    yc = cheng.g_a(x_pad)
    params = cheng.h_s(cheng.entropy_bottleneck(cheng.h_a(yc))[0])
    ctx = cheng.context_prediction(cheng.gaussian_conditional.quantize(yc, "dequantize"))
    scales, means = cheng.entropy_parameters(torch.cat((params, ctx), dim=1)).chunk(2, 1)

h, w = size
up = lambda m: np.kron(m, np.ones((16, 16)))[:h, :w]   # chaque élément latent couvre 16x16 pixels
bmap_f, bmap_c = bit_allocation_map(out["likelihoods"]["y"]), bit_allocation_map(oc["likelihoods"]["y"])
vmax = max(bmap_f.max(), bmap_c.max())
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].imshow(img); ax[0].imshow(up(bmap_f), cmap="inferno", alpha=0.6, vmin=0, vmax=vmax); ax[0].set_title("Bits : factorisé")
ax[1].imshow(img); ax[1].imshow(up(bmap_c), cmap="inferno", alpha=0.6, vmin=0, vmax=vmax); ax[1].set_title("Bits : hyperprior + contexte")
ax[2].imshow(scales.mean(1)[0].cpu(), cmap="viridis"); ax[2].set_title("σ prédit (moyenne sur les canaux)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

bits_y = (-torch.log2(oc["likelihoods"]["y"])).sum().item(); bits_z = (-torch.log2(oc["likelihoods"]["z"])).sum().item()
rc = to_uint8(crop(oc["x_hat"], size))
print(f"factorized       : {bpp_est:.3f} bpp (estimé), PSNR {psnr(img, rec):.2f} dB")
print(f"cheng2020-anchor : {(bits_y + bits_z)/n_pixels(img):.3f} bpp (estimé), PSNR {psnr(img, rc):.2f} dB")
print(f"Part du flux z (information adjacente) dans cheng2020 : {100*bits_z/(bits_y + bits_z):.1f} %")
print(f"Paramètres : factorized {n_params(net)/1e6:.1f} M | cheng2020-anchor {n_params(cheng)/1e6:.1f} M")

**❓ Questions**
5. Où chaque modèle dépense-t-il le plus de bits ? Comparez avec un codec par blocs à QP constant, puis la carte du modèle factorisé à celle de cheng2020.
6. L'hyperprior coûte des bits (flux `z`). Pourquoi est-il malgré tout rentable ? Quelle part du flux total représente `z` ?

---
# Partie C — Comparaison avec les codecs classiques (~40 min)

### Les ancres et la question de la couleur

Les codecs appris de CompressAI travaillent en **RGB** et sont évalués en **PSNR RGB**. Les codecs vidéo classiques travaillent en **YUV**, souvent **sous-échantillonné 4:2:0** : la chrominance est stockée à résolution réduite, ce qui pénalise un codec quand on mesure ensuite en RGB.

Pour séparer ce qui vient du codec et ce qui vient du format couleur, on code avec HEVC **dans les deux formats** :

| Ancre | Format | Outil |
|---|---|---|
| JPEG | YCbCr (géré par la bibliothèque) | Pillow |
| HEVC intra 4:4:4 | YUV 4:4:4 8 bits | x265 via ffmpeg |
| HEVC intra 4:2:0 | YUV 4:2:0 8 bits | x265 via ffmpeg |
| **VVC intra 4:2:0** | YUV 4:2:0, profondeur interne 10 bits | **VVenC** (Fraunhofer HHI) — ne gère pas le 4:4:4 |

Toutes les conversions RGB ↔ YUV utilisent la même matrice **BT.709 pleine échelle**. Chaque fonction renvoie `(nombre d'octets, image décodée)`, en-têtes compris.

In [ ]:
KR, KB = 0.2126, 0.0722   # BT.709

def rgb_to_yuv_planes(img, chroma="420"):
    x = img.astype(np.float64) / 255
    r, g, b = x[..., 0], x[..., 1], x[..., 2]
    y = KR * r + (1 - KR - KB) * g + KB * b
    u = (b - y) / (2 * (1 - KB)) + 0.5
    v = (r - y) / (2 * (1 - KR)) + 0.5
    if chroma == "420":
        u = u.reshape(u.shape[0] // 2, 2, u.shape[1] // 2, 2).mean((1, 3))
        v = v.reshape(v.shape[0] // 2, 2, v.shape[1] // 2, 2).mean((1, 3))
    q = lambda t: np.clip(np.round(t * 255), 0, 255).astype(np.uint8)
    return q(y), q(u), q(v)

def yuv_planes_to_rgb(Y, U, V):
    up = lambda t: t if t.shape == Y.shape else t.repeat(2, 0).repeat(2, 1)
    y, u, v = Y / 255.0, up(U) / 255.0 - 0.5, up(V) / 255.0 - 0.5
    r = y + 2 * (1 - KR) * v
    b = y + 2 * (1 - KB) * u
    g = (y - KR * r - KB * b) / (1 - KR - KB)
    return np.clip(np.round(np.stack([r, g, b], -1) * 255), 0, 255).astype(np.uint8)

def write_planes(path, planes):
    with open(path, "wb") as f:
        for p in planes: f.write(p.tobytes())

def read_planes(path, h, w, chroma):
    a = np.fromfile(path, np.uint8)
    ch, cw = (h // 2, w // 2) if chroma == "420" else (h, w)
    Y = a[:h * w].reshape(h, w); U = a[h * w:h * w + ch * cw].reshape(ch, cw); V = a[h * w + ch * cw:h * w + 2 * ch * cw].reshape(ch, cw)
    return Y, U, V

def codec_x265(img, qp, chroma="444", preset="medium"):
    h, w = img.shape[:2]
    with tempfile.TemporaryDirectory() as d:
        src, bs, out = [os.path.join(d, f) for f in ("in.yuv", "out.hevc", "out.yuv")]
        write_planes(src, rgb_to_yuv_planes(img, chroma))
        fmt = f"yuv{chroma}p"
        sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", fmt, "-s", f"{w}x{h}", "-i", src,
            "-c:v", "libx265", "-preset", preset, "-x265-params", f"qp={qp}:log-level=none", "-frames:v", "1", "-f", "hevc", bs])
        sh(["ffmpeg", "-y", "-loglevel", "error", "-i", bs, "-f", "rawvideo", "-pix_fmt", fmt, out])
        return os.path.getsize(bs), yuv_planes_to_rgb(*read_planes(out, h, w, chroma))

print("Plafond dû au seul format 4:2:0 (aucune compression) : PSNR RGB = %.2f dB"
      % psnr(img, yuv_planes_to_rgb(*rgb_to_yuv_planes(img, "420"))))
for name, f in [("JPEG q=50", lambda: codec_pil(img, "JPEG", 50)),
                ("x265 444 QP32", lambda: codec_x265(img, 32, "444")), ("x265 420 QP32", lambda: codec_x265(img, 32, "420"))]:
    n, r = f()
    print(f"{name:14s}: {8*n/n_pixels(img):.3f} bpp, PSNR {psnr(img, r):.2f} dB, MS-SSIM {msssim(img, r):.4f}")

### VVC intra avec VVenC

L'exécutable est pré-compilé sur le dépôt de l'enseignant ; à défaut, il est compilé ici (~10 min, une seule fois : il est conservé sur votre Drive). On désactive l'adaptation perceptuelle du QP (`PerceptQPA`), qui optimise une métrique visuelle et non le PSNR.

In [ ]:
VVENC_TAG = "v1.14.0"
VVENC_BIN = os.path.join(WORK, "bin", "vvencFFapp")

def setup_vvenc():
    if os.path.exists(VVENC_BIN):
        return True
    b = hf_get("bin/vvencFFapp")
    if b:
        shutil.copy(b, VVENC_BIN); os.chmod(VVENC_BIN, 0o755); return True
    print("Compilation de VVenC (~10 min)…")
    src = os.path.join(LOCAL, "vvenc")
    if not os.path.exists(src):
        sh(["git", "clone", "-q", "--depth", "1", "--branch", VVENC_TAG, "https://github.com/fraunhoferhhi/vvenc.git", src])
    sh(f"cmake -S {src} -B {src}/build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build {src}/build -j{os.cpu_count()} --target vvencFFapp > /dev/null")
    shutil.copy(glob.glob(f"{src}/bin/**/vvencFFapp", recursive=True)[0], VVENC_BIN); os.chmod(VVENC_BIN, 0o755)
    return True

def codec_vvenc(img, qp, preset="medium"):
    h, w = img.shape[:2]
    with tempfile.TemporaryDirectory() as d:
        src, bs, rec = [os.path.join(d, f) for f in ("in.yuv", "out.266", "rec.yuv")]
        write_planes(src, rgb_to_yuv_planes(img, "420"))
        sh([VVENC_BIN, f"--preset={preset}", f"--InputFile={src}", f"--SourceWidth={w}", f"--SourceHeight={h}",
            "--InputBitDepth=8", "--OutputBitDepth=8", "--InternalBitDepth=10", "--FrameRate=25",
            "--FramesToBeEncoded=1", "--IntraPeriod=1", f"--QP={qp}", "--PerceptQPA=0", "--Verbosity=1",
            f"--Threads={os.cpu_count()}", f"--BitstreamFile={bs}", f"--ReconFile={rec}"])
        return os.path.getsize(bs), yuv_planes_to_rgb(*read_planes(rec, h, w, "420"))

VVENC_OK = False
if RUN_VVENC:
    try:
        VVENC_OK = setup_vvenc()
        n, r = codec_vvenc(img, 32)
        print(f"VVC intra QP32 : {8*n/n_pixels(img):.3f} bpp, PSNR {psnr(img, r):.2f} dB")
    except Exception as e:
        VVENC_OK = False; print("VVenC indisponible :", str(e)[:600])

### Courbes débit-distorsion sur l'ensemble de test

**Remarque sur les modèles autorégressifs.** `cheng2020-anchor` utilise un **modèle de contexte autorégressif** : la probabilité de chaque élément latent dépend des éléments déjà décodés. Son décodage est **séquentiel** et très lent dans CompressAI (plusieurs dizaines de secondes par image) : nous mesurons son débit **estimé** (exercice 3), dont vous avez vérifié qu'il est très proche du débit réel.

⏱️ 8 à 15 minutes sur GPU T4, téléchargement des poids compris. Les résultats sont **sauvegardés codec par codec** : en cas de coupure, la cellule reprend où elle s'était arrêtée.

In [ ]:
QUALITIES = [1, 2, 3, 4, 5, 6]
LEARNED = {
    "bmshj2018-factorized": (bmshj2018_factorized, True),
    "cheng2020-anchor":     (cheng2020_anchor, False),   # False : débit estimé
}
CLASSICAL = {
    "JPEG":              ([10, 20, 30, 45, 60, 75, 85, 92], lambda im, p: codec_pil(im, "JPEG", p)),
    "x265 intra 444":    ([22, 27, 32, 37, 42, 47], lambda im, p: codec_x265(im, p, "444")),
    "x265 intra 420":    ([22, 27, 32, 37, 42, 47], lambda im, p: codec_x265(im, p, "420")),
}
if VVENC_OK:
    CLASSICAL["VVC intra 420 (VVenC)"] = ([22, 27, 32, 37, 42, 47], codec_vvenc)

def sync():
    if device == "cuda": torch.cuda.synchronize()

def eval_learned(model, images, real_coding=True):
    rows = []
    for im in images:
        x, size = pad(to_tensor(im).to(device))
        with torch.no_grad():
            if real_coding:
                e, x_hat, t_enc, t_dec, gap = roundtrip(model, x)
                if gap > 0.05:
                    print(f"  ⚠️ décodage incohérent (écart {gap:.2f}) — résultat faux pour cette image")
                bpp = 8 * sum(len(s[0]) for s in e["strings"]) / n_pixels(im)
            else:
                o = model(x); bpp = estimated_bpp(o, n_pixels(im)).item()
                x_hat, t_enc, t_dec = o["x_hat"], np.nan, np.nan
        r = to_uint8(crop(x_hat, size))
        rows.append(dict(bpp=bpp, psnr=psnr(im, r), msssim=msssim(im, r), t_enc=t_enc, t_dec=t_dec))
    return pd.DataFrame(rows).mean().to_dict()

def eval_classical(fn, p, images):
    rows = []
    for im in images:
        t0 = time.time(); n, r = fn(im, p); t = time.time() - t0
        rows.append(dict(bpp=8 * n / n_pixels(im), psnr=psnr(im, r), msssim=msssim(im, r), t_enc=t))
    return pd.DataFrame(rows).mean().to_dict()

RD_CSV = os.path.join(WORK, "results", "tp1_rd.csv")
df = pd.read_csv(RD_CSV) if os.path.exists(RD_CSV) else pd.DataFrame(columns=["codec", "param"])
def have(codec): return codec in set(df.codec)
def add(rows):
    global df
    df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True); df.to_csv(RD_CSV, index=False)

for name, (builder, real) in LEARNED.items():
    if have(name): continue
    rows = []
    for q in QUALITIES:
        m = builder(quality=q, pretrained=PRETRAINED).eval().to(device); m.update(force=True)
        r = eval_learned(m, test_imgs, real); r.update(codec=name, param=q, params_M=n_params(m) / 1e6)
        rows.append(r); print(f"{name:22s} q={q}: {r['bpp']:.3f} bpp, {r['psnr']:.2f} dB"); del m
    add(rows)
for name, (params, fn) in CLASSICAL.items():
    if have(name): continue
    add([{**eval_classical(fn, p, test_imgs), "codec": name, "param": p} for p in params]); print(name, "ok")

if USE_PRECOMPUTED_IF_MISSING and not VVENC_OK:
    f = hf_get("precomputed/tp1_rd.csv")
    if f:
        pre = pd.read_csv(f); pre = pre[pre.codec.str.startswith("VVC")]
        pre["codec"] = pre.codec + " (pré-calculé)"; df = pd.concat([df, pre], ignore_index=True)
df.round(4).head()

In [ ]:
def style(c):
    return {"JPEG": "s--", "WebP": "^--", "x265 intra 444": "D--", "x265 intra 420": "d:"}.get(c, "P--" if c.startswith("VVC") else "o-")
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for codec, g in df.groupby("codec", sort=False):
    g = g.sort_values("bpp")
    ax[0].plot(g.bpp, g.psnr, style(codec), label=codec, ms=4)
    ax[1].plot(g.bpp, msssim_db(g.msssim), style(codec), label=codec, ms=4)
ax[0].set_ylabel("PSNR RGB (dB)"); ax[1].set_ylabel("MS-SSIM (dB)")
for a in ax:
    a.set_xlabel("débit (bpp)"); a.set_xlim(0, 1.6); a.grid(alpha=.3)
ax[0].legend(fontsize=8); ax[0].set_ylim(24, 40)
plt.suptitle("Courbes débit-distorsion — Kodak 17-24"); plt.tight_layout(); plt.show()

### BD-rate (Bjøntegaard delta rate)

Le **BD-rate** résume l'écart entre deux courbes en un seul nombre : la **variation moyenne de débit à qualité égale** (en %). Négatif = économie de débit par rapport à l'ancre. On interpole $\log(\text{débit})$ en fonction de la qualité (interpolation monotone PCHIP, comme les outils de référence) et on intègre sur l'intervalle de qualité commun.

In [ ]:
def bd_rate(rate_anchor, q_anchor, rate_test, q_test):
    def interp(rate, q):
        q, lr = np.asarray(q, float), np.log(np.asarray(rate, float)); i = np.argsort(q)
        return PchipInterpolator(q[i], lr[i])
    lo, hi = max(min(q_anchor), min(q_test)), min(max(q_anchor), max(q_test))
    if hi <= lo: return np.nan
    fa, ft = interp(rate_anchor, q_anchor), interp(rate_test, q_test)
    return (np.exp((ft.integrate(lo, hi) - fa.integrate(lo, hi)) / (hi - lo)) - 1) * 100

def bd_table(anchor):
    a = df[df.codec == anchor]
    return pd.DataFrame([dict(codec=c, **{
        "BD-rate PSNR (%)": bd_rate(a.bpp, a.psnr, g.bpp, g.psnr),
        "BD-rate MS-SSIM (%)": bd_rate(a.bpp, msssim_db(a.msssim), g.bpp, msssim_db(g.msssim))})
        for c, g in df.groupby("codec", sort=False)]).set_index("codec").round(1)

for anchor in ["x265 intra 444"] + [c for c in df.codec.unique() if c.startswith("VVC")][:1]:
    print("Ancre :", anchor); display(bd_table(anchor))

In [ ]:
cx = (df.groupby("codec", sort=False).agg(params_M=("params_M", "mean"), t_enc_s=("t_enc", "mean"), t_dec_s=("t_dec", "mean")).round(3))
cx["matériel"] = [("CPU (codage) / " + device.upper()) if c in LEARNED else "CPU" for c in cx.index]
print("Temps moyen par image (classiques : encodage + décodage + écriture de fichiers, sur CPU)")
cx

**❓ Questions**
7. Comparez **x265 4:4:4** et **x265 4:2:0** : combien coûte le sous-échantillonnage de la chrominance en BD-rate mesuré en RGB ? Quelle ancre est la plus juste face aux codecs appris ?
8. Classez les codecs par BD-rate PSNR, puis MS-SSIM. Le classement change-t-il ? Sur quelle plage de débits les codecs appris sont-ils le plus compétitifs ?
9. Pourquoi le décodage de `cheng2020-anchor` est-il séquentiel, et comment les modèles récents (damier, par canaux, JPEG AI) contournent-ils ce problème ?

---
# Partie D — Comparaison visuelle à bas débit (~10 min)

À très bas débit, les artefacts sont de **nature différente** : effets de blocs et *ringing* pour les codecs par transformée de blocs, flou et textures lissées pour les codecs appris.

In [ ]:
TARGET_BPP = 0.15
img = test_imgs[6]

def closest(cands, target):
    best = None
    for p, f in cands:
        n, r = f(p); b = 8 * n / n_pixels(img)
        if best is None or abs(b - target) < abs(best[0] - target): best = (b, r, p)
    return best

def learned_codec(builder, real=True):
    def f(q):
        m = builder(quality=q, pretrained=PRETRAINED).eval().to(device)
        x, s = pad(to_tensor(img).to(device))
        if real:
            m.update(force=True)
            e, xh, _, _, _ = roundtrip(m, x); r = to_uint8(crop(xh, s))
            return sum(len(t[0]) for t in e["strings"]), r
        with torch.no_grad():
            o = m(x)                      # reconstruction identique au décodage ; débit estimé
        return estimated_bpp(o, n_pixels(img)).item() / 8 * n_pixels(img), to_uint8(crop(o["x_hat"], s))
    return f

cands = {
    "JPEG":            [(q, lambda q: codec_pil(img, "JPEG", q)) for q in range(2, 40, 2)],
    "x265 intra 444":  [(qp, lambda qp: codec_x265(img, qp, "444")) for qp in range(30, 51, 2)],
    "factorized":      [(q, learned_codec(bmshj2018_factorized)) for q in [1, 2, 3]],
    "cheng2020 (estimé)": [(q, learned_codec(cheng2020_anchor, real=False)) for q in [1, 2, 3]],
}
if VVENC_OK:
    cands["VVC intra 420"] = [(qp, lambda qp: codec_vvenc(img, qp)) for qp in range(30, 51, 2)]
crop_box = (slice(60, 260), slice(330, 630))
fig, axes = plt.subplots(1, len(cands) + 1, figsize=(3.4 * (len(cands) + 1), 3.6))
axes[0].imshow(img[crop_box]); axes[0].set_title("Original")
for ax, (name, lst) in zip(axes[1:], cands.items()):
    b, r, p = closest(lst, TARGET_BPP)
    ax.imshow(r[crop_box]); ax.set_title(f"{name}\n{b:.3f} bpp, {psnr(img, r):.1f} dB", fontsize=9)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

**❓ Question**
10. Décrivez les artefacts de chaque codec. Le PSNR reflète-t-il votre jugement visuel ? Quelles métriques (MS-SSIM, VMAF, LPIPS…) seraient plus adaptées, et avec quelles limites ? *(Ce sera le cœur de la séance 3.)*

---
# Partie E — Fine-tuning : déplacer un modèle pré-entraîné sur la courbe R-D (~25 min)

Entraîner un codec de l'état de l'art prend des jours de GPU. En pratique, on part d'un modèle pré-entraîné et on l'**affine** pour un nouveau besoin : un autre compromis débit-distorsion, une autre métrique, ou un type de contenu.

On part de `bmshj2018-factorized` **qualité 3** ($\lambda = 0.0067$) et on l'affine avec le $\lambda$ de la **qualité 4** ($\lambda = 0.0130$), sur des crops des images 1 à 16.

Valeurs de λ (MSE) de CompressAI : q1 0.0018 · q2 0.0035 · q3 0.0067 · q4 0.0130 · q5 0.0250 · q6 0.0483 · q7 0.0932 · q8 0.1800.

### ✏️ Exercice 5 — Fonction de coût débit-distorsion

Implémentez $L = R + \lambda \cdot 255^2 \cdot \mathrm{MSE}(x, \hat x)$, avec $R$ le débit estimé en bpp (exercice 3, moyenné par pixel du batch) et $\hat x$ = `out["x_hat"]`. Renvoyez `{"loss", "bpp", "mse"}`. C'est la même perte qu'en partie A, appliquée à un vrai modèle.

In [ ]:
def rd_loss(out, x, lmbda):
    N, _, H, W = x.shape
    # TODO : débit estimé (bpp), MSE, perte totale
    raise NotImplementedError("Exercice 5")

In [ ]:
LMBDA_NEW = 0.0130
N_ITERS, BATCH, FT_PATCH = (400, 8, 256) if device == "cuda" else (60, 4, 256)

ft = bmshj2018_factorized(quality=3, pretrained=PRETRAINED).to(device).train()
main_params = [p for n, p in ft.named_parameters() if not n.endswith(".quantiles")]
aux_params  = [p for n, p in ft.named_parameters() if n.endswith(".quantiles")]
opt, aux_opt = torch.optim.Adam(main_params, lr=1e-4), torch.optim.Adam(aux_params, lr=1e-3)

log = []
for it in range(N_ITERS):
    xb = random_patches(BATCH, FT_PATCH)
    if np.random.rand() < .5: xb = xb.flip(-1)
    opt.zero_grad(); aux_opt.zero_grad()
    L = rd_loss(ft(xb), xb, LMBDA_NEW)
    L["loss"].backward()
    torch.nn.utils.clip_grad_norm_(main_params, 1.0)
    opt.step()
    aux = ft.aux_loss(); aux.backward(); aux_opt.step()
    log.append({k: v.item() for k, v in L.items()})
    if it % 100 == 0 or it == N_ITERS - 1:
        print(f"it {it:4d} | loss {L['loss'].item():.3f} | bpp {L['bpp'].item():.3f} | PSNR {10*np.log10(1/L['mse'].item()):.2f} dB | aux {aux.item():.1f}")

lg = pd.DataFrame(log).rolling(20, min_periods=1).mean()
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(lg.bpp); ax[0].set_title("bpp (entraînement)"); ax[1].plot(10 * np.log10(1 / lg.mse)); ax[1].set_title("PSNR (entraînement)")
plt.tight_layout(); plt.show()

In [ ]:
ft.eval(); ft.update(force=True)
r_ft = eval_learned(ft, test_imgs, real_coding=True)
print(f"Modèle affiné : {r_ft['bpp']:.3f} bpp, {r_ft['psnr']:.2f} dB")

g = df[df.codec == "bmshj2018-factorized"].sort_values("bpp")
plt.figure(figsize=(7, 4.5))
plt.plot(g.bpp, g.psnr, "o-", label="factorized pré-entraîné (q1–q6)")
for _, row in g.iterrows():
    plt.annotate(f"q{int(row.param)}", (row.bpp, row.psnr), textcoords="offset points", xytext=(5, -10), fontsize=8)
plt.plot(r_ft["bpp"], r_ft["psnr"], "r*", ms=14, label=f"q3 affiné avec λ={LMBDA_NEW}")
plt.xlabel("bpp"); plt.ylabel("PSNR (dB)"); plt.grid(alpha=.3); plt.legend(); plt.xlim(0, 1.0)
plt.title("Fine-tuning : déplacement sur la courbe R-D"); plt.show()

**❓ Questions**
11. Le modèle affiné s'est-il déplacé dans la direction attendue ? Atteint-il la courbe du modèle q4 ? Qu'est-ce qui limite ce fine-tuning (données, itérations, architecture fixe) ?
12. À quoi sert la perte auxiliaire `aux_loss()` ? Que se passerait-il si on oubliait `update()` avant `compress()` ? (Faites le lien avec l'exercice 2.)
13. **Bonus** : affinez pour **MS-SSIM** avec $L = R + \lambda (1 - \text{MS-SSIM})$ (λ ≈ 8.73 pour la qualité 3), ou sur un type de contenu particulier (captures d'écran, visages…), et mesurez le gain par rapport au modèle générique.

---
## Pour aller plus loin
- **JPEG AI** (ISO/IEC 6048) : premier standard de codage d'images appris, avec un logiciel de référence public.
- Autres modèles du zoo CompressAI : `cheng2020-attn` (avec modules d'attention) et le modèle vidéo `ssf2020` (séance 2).

## Rendu
Rapport d'équipe : courbes R-D, tableaux BD-rate et complexité, réponses argumentées aux questions 1 à 12, et une conclusion : **dans quels cas recommanderiez-vous aujourd'hui un codec d'images appris plutôt que HEVC ou VVC intra ?**